# Cache quality audit
Run in the submission directory. This notebook checks real cached data at the contract/date grain; all candidate requests completed, but the cache is not a verified full listing universe.

In [ ]:
import pandas as pd
from fetch_options_v2 import to_long
p = pd.read_pickle('option_pipeline_data_v2.pkl')
d = to_long(p['options'])
observed = d[d[['MID_PRICE', 'TRDPRC_1']].notna().any(axis=1)]
assert not d.duplicated(['date', 'ric']).any()
assert not (d[['MID_PRICE', 'TRDPRC_1']] < 0).any().any()
assert (pd.to_datetime(d['expiry']) >= d['date']).all()
display(observed.groupby('type').agg(contracts=('ric', 'nunique'), observations=('ric', 'size')))

In [ ]:
first = observed.groupby('ric')['date'].min()
contracts = d[['ric', 'expiry', 'type']].drop_duplicates().set_index('ric')
contracts['first_seen'] = first
date = pd.Timestamp('2026-07-10')
side = 'Call'
eligible = contracts[(contracts['type'] == side) & (contracts['first_seen'] <= date) & (pd.to_datetime(contracts['expiry']) >= date)]
day = observed[(observed['date'] == date) & (observed['type'] == side)]
mid_only = day.MID_PRICE.notna() & day.TRDPRC_1.isna()
both = day.MID_PRICE.notna() & day.TRDPRC_1.notna()
print('Sampled active series:', len(eligible))
print('Mid without trade (%):', 100 * mid_only.sum() / len(eligible) if len(eligible) else None)
print('Paired median gap:', (day.loc[both, 'MID_PRICE'] - day.loc[both, 'TRDPRC_1']).abs().median())

Interpretation: no duplicate contract/date keys or negative prices were found. The returned price sample contains both calls and puts. First observed price through expiry is a disclosed listing proxy; the exact listed-universe percentage cannot be established. Request failures must remain distinct from market sparsity. The UI and README describe remaining sampling limits, Friday-only enumeration, absent corporate-action verification and the inability to infer executable fills from daily data.